In [5]:
# Standard library imports
import asyncio
# Third-party imports for MCP (Model Context Protocol) and LangGraph
from langchain_mcp_adapters.client import MultiServerMCPClient # Connects to MCP servers
# from langchain.agent import create_react_agent # Creates ReAct-style agents
from langchain.agents import create_agent # Creates ReAct-style agents
from langgraph.checkpoint.memory import InMemorySaver # Provides conversation memory
from langchain_openai import ChatOpenAI # OpenAI chat model integration


In [6]:
async def main():
    """
    Main function that sets up and runs an AI agent with access to multiple MCP servers.
    The agent can access Context7 library documentation and Met Museum collections.
    """


Multi-server MCP Client
This client acts as the bridge between our application and external services, handling complex communication protocols (MCP) automatically. MCP supports multiple transport mechanisms, each optimized for different deployment scenarios.

Context7 MCP server: Library documentation via HTTP transport

Streamable HTTP transport enables real-time communication with remote servers

Stateless connection: each request is independent, perfect for cloud services

Cross-platform compatibility works from any environment with internet access

Metropolitan Museum of Art MCP server: Art collection data via Node.js STDIO

STDIO transport creates a direct process-to-process communication channel

Local execution runs the MCP server as a subprocess on the same machine

Persistent connection maintains state throughout the session

Lower latency since no network overhead, ideal for local tools and data sources

MCP protocol enables secure tool and data access

Standardized interface: same client code works with different transport types

Tool discovery is handled automatically regardless of transport method

Type-safe tool definitions with automatic schema validation

The client abstracts away the complexity of connecting to different server types and protocols.

In [7]:
	# Configure MCP (Model Context Protocol) servers
    # These servers provide tools that the AI agent can use
client = MultiServerMCPClient(
        {
            # Context7 server - provides access to library documentation
            "context7": {
                "url": "https://mcp.context7.com/mcp",        # Server endpoint
                "transport": "streamable_http",                # Communication protocol
            },
            # Met Museum server - provides access to museum collection data
            "met-museum": {
                "command": "npx",                              # Node.js package runner
                "args": ["-y", "metmuseum-mcp"],              # Install and run met museum MCP
                "transport": "stdio",                         # Communication via stdin/stdout
            }
        }
    )


### ReAct Agent Configuration
Here we assemble the intelligence layer that will power our agent's capabilities.

ChatOpenAI, the OpenAI LLM provides reasoning and language understanding

Tool retrieval from MCP servers for external interactions

InMemorySaver maintains conversation context across interactions

Thread configuration groups messages for session continuity

This configuration enables natural, context-aware conversations rather than isolated exchanges.

In [ ]:
	# Initialize the OpenAI language model
    # This model will power the AI agent's reasoning and responses
openai_model = ChatOpenAI(
        model="gpt-5-nano",  # Using OpenAI's GPT-5 Nano model
    )

	# Initialize the Watsonx language model
    # This model will power the AI agent's reasoning and responses if the OpenAI model gets rate limited. Uncomment and use it. 

    #watsonx_model = ChatWatsonx(
    #    model_id="ibm/granite-4-h-small",
    #    url="https://us-south.ml.cloud.ibm.com",
    #    project_id="skills-network"
    #)
    

    # Retrieve all available tools from the configured MCP servers
    # These tools allow the agent to interact with external services
tools = await client.get_tools()

    # Set up conversation memory using InMemorySaver
    # This allows the agent to remember previous messages in the conversation
checkpointer = InMemorySaver()

	# Configuration for conversation persistence
    # The thread_id ensures all messages in this session are grouped together
config = {"configurable": {"thread_id": "conversation_id"}}


### ReAct Agent
The ReAct agent represents the core intelligence of our application, combining reasoning with action.

ReAct = Reasoning + Acting: thinks through problems then takes action

Multi-step problem solving using available tools

Active information gathering through MCP server tools

More powerful than simple chatbots that can only respond
This agent can both understand complex requests and actively fulfill them through tool usage.

In [ ]:
	# Create the ReAct agent with all components
    # ReAct = Reasoning + Acting (agent can reason about and use tools)
agent = create_agent(
        model=openai_model,         # The language model to use, replace with watsonx_model if you receive rate limiting errors
        tools=tools,                # Available tools from MCP servers
        checkpointer=checkpointer   # Memory system for conversation history
    )


In [ ]:
# Create server parameters for stdio connection
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

from langchain_mcp_adapters.tools import load_mcp_tools
# from langgraph.prebuilt import create_react_agent

server_params = StdioServerParameters(
    command="python",
    # Make sure to update to the full absolute path to your math_server.py file
    args=["/path/to/math_server.py"],
)

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        # Initialize the connection
        await session.initialize()

        # Get tools
        tools = await load_mcp_tools(session)

        # Create and run the agent
        agent = create_agent("openai:gpt-4.1", tools)
        agent_response = await agent.ainvoke({"messages": "what's (3 + 5) x 12?"})


### This is a lower-level approach that uses modules:

StdioServerParameters: Parameter schema for a stdio server; command and args required

stdio_client: Takes the transport parameters and creates an asynchronous client connected to the server with read/write streams so the client can receive JSON messages over the specified transport: standard I/O

ClientSession: Handles all requests and messages over the input/ouput streams, this includes initializing the handshake, sending requests to the server, receiving responses, managing session states, and cleanup

The streamable HTTP version of this looks something like the following:

In [ ]:
# Use server from examples/servers/streamable-http-stateless/

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

# from langgraph.prebuilt import create_react_agent
from langchain_mcp_adapters.tools import load_mcp_tools

async with streamable_http_client("http://localhost:3000/mcp") as (read, write, _):
    async with ClientSession(read, write) as session:
        # Initialize the connection
        await session.initialize()

        # Get tools
        tools = await load_mcp_tools(session)
        agent = create_agent("openai:gpt-4.1", tools)
        math_response = await agent.ainvoke({"messages": "what's (3 + 5) x 12?"})
